# 02. Branched morphologies

In this tutorial, we will explore how to use AxonML to simulate branched neuron morphologies. Branched morphologies are essential for accurately modeling the complex structures of real neurons, which often have multiple dendritic branches and axonal projections.


## General principles

Branched morphologies inherit the same "array of compartments in a {py:class}`~axonml.models.core.Population`" abstraction as unbranched morphologies. Each compartment can still be treated as a single electrical unit, with its own membrane potential and ionic currents. The key difference is that branched morphologies allow for more complex connectivity between compartments, enabling the simulation of signal propagation through dendritic trees and axonal arbors.

To define a branched morphology in AxonML, we use the {py:class}`~axonml.models.tree.Tree` class, which allows us to specify the structure of the neuron using a tree-like representation. We use NEURON as a frontend to define the morphology and then import it into AxonML, enabling users to recreate complex neuronal structures and reuse complex algorithmic modifications (e.g., to myelinate neurons) with ease.

## Example: 3D morpholgy with HH-dynamics

We will simulate a single neuron with a branched morphology using Hodgkin-Huxley dynamics. AxonML implements variosu convenience functions to read in morphologies from standard file formats like SWC or from NEURON.

In [ ]:
import axonml as ax
from axonml.models.mod import hh

model = ax.Tree.from_asc(
    "example.asc", 
    v_init=-65.0, 
    celsius=6.3,
    rhoa=100.0,
    principal_axis=[0, 1, 0]
)  # Load morphology from file

# this automatically applies the D-lambda rule to set compartment lengths, with
# d_lambda=0.1 @ 100 Hz. You can adjust this as needed, e.g.:
# model = ax.Tree.from_asc("example.asc", d_lambda=0.05, freq=200.0)

# The default axial resistivity and membrane capacitance is 35.4 Ω·cm 
# and 1.0 µF/cm² (consistent with NEURON), but you can change this with the 
# 'rhoa' and 'cm' arguments if desired.

# Note also that the principal_axis argument can be used to specify the main
# orientation of the tree morphology. Here we specify that the principal axis
# is along the y-axis (default is z-axis, [0, 0, 1]).

Let's visualize:

In [ ]:
from axonml.models.visualization import vis_2d

vis_2d(model, view="x")
vis_2d(model, view="y")
vis_2d(model, view="z")

One thing to note at this point is that AxonML automatically constructs labeled {py:class}`~axonml.models.slice.Slice`'s for branched morphologies, which can be used to easily reference specific parts of the morphology (e.g., soma, dendrites, axon). This is particularly useful for applying stimuli or recording from specific regions of the neuron:

In [ ]:
print(model.soma.v)
print(model.dend.v)

For modeling extracellular stimulation, it is especially important to be able to control where and in what orientiation the morphology is placed in 3D space. AxonML provides several convenience functions to help with this, such as {py:meth}`~axonml.models.tree.Tree.move_to` to move the soma, {py:meth}`~axonml.models.tree.Tree.recentre` to center the morphology at the origin (equivalent to moving the soma to [0, 0, 0]), and {py:meth}`~axonml.models.tree.Tree.rotate_into_direction` to align the morphology along a specified axis.

In [ ]:
print(model.soma.x.item(), model.soma.y.item(), model.soma.z.item())
model.move_to(100.0, 0, 0)
print(model.soma.x.item(), model.soma.y.item(), model.soma.z.item())
model.recentre()
print(model.soma.x.item(), model.soma.y.item(), model.soma.z.item())

In [ ]:
model = model.rotate_into_direction([[0, 0, 1]])  # Rotate to align along z-axis

for axis in ["x", "y", "z"]:
    vis_2d(model, view=axis)

The exact same API is used to specify biophysical properties, stimuli, and recording as for unbranched morphologies. For example, we can insert Hodgkin-Huxley dynamics into the entire morphology using the {py:meth}`~axonml.models.core.Population.insert` method:

In [ ]:
model.insert(hh)

And then stimulate e.g. with a current source injected into the soma:

In [ ]:
from axonml.units import nA
model.soma.inject(ax.mono_rect(amp=2.0 * nA, pw=1.0, delay=0.1))

rec = ax.callbacks.RecorderLambda(
    {
        "soma": lambda cell: cell.soma.v,
        "dend": lambda cell: cell.dend[0, -1].v,
        "axon": lambda cell: cell.axon[0, -1].v,
    }
)

tstop, dt = 10.0, 0.025

rec.reset()
model.initialize()
model.run(tstop=10.0, dt=0.025, callbacks=[rec])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
t = np.arange(0, tstop+dt, dt)
plt.figure(figsize=(5, 3), dpi=200)
plt.plot(t, rec.numpy('soma').flatten(), label="soma")
plt.plot(t, rec.numpy('dend').flatten(), label="dend[0, -1]")
plt.plot(t, rec.numpy('axon').flatten(), label="axon[0, -1]")
plt.xlabel("Time (ms)")
plt.ylabel("Membrane potential (mV)")
plt.legend()
plt.show()

The exact same API for extracellular stimulation is also valid:

In [ ]:
from axonml.units import mA, kHz

stim_source = ax.isotropic_point(x=400.0, y=-400.0, z=200.0)  # point source at (400, -400, 200) * um
ve_stim = stim_source(model)
i_stim = ax.sin(amp=0.5 * mA, freq=1.0 * kHz)  # 0.05 mA, 1 kHz sine wave

tstop, dt = 50.0, 0.001

# run as before, but supply extracellular potential for stimulation
# for longer simulations, we recommend using longrun with chunking
rec.reset()
model.delete_injections() # remove all previous current injection(s)
model.initialize()
model.longrun(
    tstop=tstop, dt=dt, chunklength=100, 
    extra=(ve_stim, i_stim), callbacks=[rec],
    progressbar=False
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

t = np.arange(0, tstop+dt, dt)
plt.figure(figsize=(5, 3), dpi=200)
plt.plot(t, rec.numpy('soma').flatten(), label="soma")
plt.plot(t, rec.numpy('dend').flatten(), label="dend[0, -1]")
plt.plot(t, rec.numpy('axon').flatten(), label="axon[0, -1]")
plt.xlabel("Time (ms)")
plt.ylabel("Membrane potential (mV)")
plt.legend()
plt.show()

And, as before, we can use the same API for LFP recording, and stimulating and recording from populations of branched morphologies as well. To get a population, we use the same {py:meth}`~axonml.models.tree.Tree.from_NEURON` or {py:meth}`~axonml.models.tree.Tree.from_swc` methods, but specify `N>1`:

```python
with ax.ctx(IMEM=1): # to record membrane currents for LFP
    pop = ax.Tree.from_asc(
        "path/to/morphology.asc",
        N=100,
    )
```
The rest of the simulation code remains unchanged.